# EquityRank · Xếp hạng cổ phiếu JPX

Dùng dữ liệu đến cuối phiên `t` để xếp hạng theo lợi nhuận từ đóng cửa `t+1` đến đóng cửa `t+2`.

[Dữ liệu, quyết định và kết quả](../reports/jpx_release.md) · [Khám phá giá](experiments/eda_stock_prices.ipynb) · [Công bố tài chính](experiments/financial_disclosures.ipynb) · [Mô hình](experiments/model_comparison.ipynb) · [Chi phí](experiments/liquidity_costs.ipynb) · [Tái lập](../reports/reproduce.md)

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file()
            and (p / "src/__init__.py").is_file())
sys.path.insert(0, str(root))

from src.diagnostics import paired_calendar_intervals

results = root / "artifacts"
if not (results / "development_summary.csv").is_file():
    raise FileNotFoundError("Tạo các bảng kết quả theo reports/reproduce.md trước khi chạy notebook.")

selection = json.loads((results / "selected_model.json").read_text(encoding="utf-8"))
winner, ridge = selection["name"], selection["baseline"]
reversal = "Intraday reversal"
model_labels = {
    "LightGBM finance": "LightGBM giá và công bố tài chính",
    "Ridge six": "Ridge với 6 đặc trưng kỹ thuật",
    "LambdaRank two sides": "LambdaRank hai vế",
    reversal: "Đảo chiều trong phiên",
}

## Tập xác thực qua sáu lượt walk-forward

- Tính spread trước phí giữa 200 mã đầu và 200 mã cuối. Thiếu `Target` ở một mã đã chọn thì bỏ spread của phiên đó.
- Chọn mô hình theo Sharpe kiểu JPX trên các phiên chấm được cho mọi ứng viên.

In [2]:
summary = pd.read_csv(results / "development_summary.csv").set_index("Model")
fold_calendar = pd.concat([
    pd.read_csv(path, parse_dates=["Date"]).assign(Window=path.parent.name)
    for path in sorted((results / "folds").glob("*/calendar.csv"))
], ignore_index=True).set_index("Date")["Window"]
validation_daily = pd.read_csv(results / "development_daily.csv", parse_dates=["Date"])
validation_spread = validation_daily.pivot(index="Date", columns="Model", values="Spread").reindex(fold_calendar.index)
common_validation = validation_spread[selection["candidates"]].notna().all(axis=1)
session_counts = common_validation.groupby(fold_calendar).agg(["size", "sum"])
session_counts.index.name = "Bắt đầu cửa sổ"
session_counts.columns = ["Có t+2 trong cửa sổ", "Chấm chung"]
session_counts.loc["Tổng"] = session_counts.sum()
assert common_validation.sum() == selection["validation_score_sessions"]
display(session_counts)

shown = list(dict.fromkeys([winner, ridge, "LambdaRank two sides", reversal]))
display(summary.loc[shown, ["JPX Sharpe", "Spread bps", "IC", "Common score sessions"]]
        .rename(index=model_labels, columns={"JPX Sharpe": "Sharpe kiểu JPX",
                "Spread bps": "Spread (bps/phiên)", "Common score sessions": "Số phiên chấm"})
        .round(4))

,Có t+2 trong cửa sổ,Chấm chung
Bắt đầu cửa sổ,,
2019-01-01,115,115
2019-07-01,122,122
2020-01-01,117,117
2020-07-01,122,121
2021-01-01,119,119
2021-07-01,106,106
Tổng,701,700


,Sharpe kiểu JPX,Spread (bps/phiên),IC,Số phiên chấm
Model,,,,
LightGBM giá và công bố tài chính,0.1252,6.0520,-0.0006,700
Ridge với 6 đặc trưng kỹ thuật,0.0768,10.3911,-0.0070,700
LambdaRank hai vế,0.0198,1.0754,0.0019,700
Đảo chiều trong phiên,0.0534,5.0968,0.0041,700


- Chấm được 700 trong 701 phiên đúng ranh giới `t+2`. Phiên 01/10/2020 thiếu giá và khối lượng để xếp hạng.
- LightGBM giá và công bố tài chính có Sharpe cao nhất trong nhóm ứng viên được dùng để chọn mô hình.

## Tập kiểm thử từ 09/12/2021

Có 130 phiên tín hiệu với `t+2` nằm trong dữ liệu. Kết quả kiểm thử đã từng được xem ở vòng trước.

In [3]:
test_summary = pd.read_csv(results / "historical_summary.csv").set_index("Model")
test_daily = pd.read_csv(results / "historical_daily.csv", parse_dates=["Date"])
test_spread = test_daily.pivot(index="Date", columns="Model", values="Spread")
common_test = test_spread.notna().all(axis=1)
assert test_summary["Common sessions"].eq(common_test.sum()).all()
test_summary["Spread std bps"] = test_spread.loc[common_test].std().mul(10_000)

display(test_summary.loc[list(dict.fromkeys([winner, ridge, reversal])), [
    "Scored sessions", "Common sessions", "JPX Sharpe", "Spread bps", "Spread std bps", "IC",
]].rename(index=model_labels, columns={"Scored sessions": "Phiên chấm riêng",
    "Common sessions": "Phiên chấm chung", "JPX Sharpe": "Sharpe kiểu JPX",
    "Spread bps": "Spread (bps/phiên)", "Spread std bps": "Độ lệch chuẩn spread (bps)",
}).round(4))

,Phiên chấm riêng,Phiên chấm chung,Sharpe kiểu JPX,Spread (bps/phiên),Độ lệch chuẩn spread (bps),IC
Model,,,,,,
LightGBM giá và công bố tài chính,129,123,0.0194,1.0475,54.0602,-0.0080
Ridge với 6 đặc trưng kỹ thuật,125,123,-0.0726,-16.9403,233.2409,-0.0485
Đảo chiều trong phiên,130,123,0.0178,2.2207,124.8990,-0.0059


- Phiên chấm riêng đếm theo từng mô hình. Chỉ số trong bảng dùng 123 phiên chấm chung với các đối chứng.
- LightGBM có Sharpe gần bằng quy tắc đảo chiều trong phiên, với spread trung bình và độ dao động đều thấp hơn.

In [4]:
comparisons = [
    ("Xác thực", validation_spread[[winner, reversal]].where(common_validation), fold_calendar),
    ("Kiểm thử", test_spread[[winner, reversal]].where(common_test),
     pd.Series("test", index=test_spread.index)),
]
comparison_rows = []
for split_name, spreads, calendar in comparisons:
    sharpe = spreads.mean() / spreads.std()
    interval = paired_calendar_intervals(spreads, calendar, reversal)
    comparison_rows.append(interval.loc[interval["Model"].eq(winner)].assign(
        Split=split_name, Delta=sharpe[winner] - sharpe[reversal],
    ))
display(pd.concat(comparison_rows, ignore_index=True)[
    ["Split", "Common score sessions", "Block", "Delta", "Low", "High"]
].rename(columns={"Split": "Tập", "Common score sessions": "Phiên chung", "Block": "Khối (phiên)",
    "Delta": "Chênh lệch Sharpe", "Low": "Cận dưới 95%", "High": "Cận trên 95%",
}).round(4))

,Tập,Phiên chung,Khối (phiên),Chênh lệch Sharpe,Cận dưới 95%,Cận trên 95%
0,Xác thực,700,5,0.0719,-0.0133,0.1666
1,Xác thực,700,10,0.0719,-0.0110,0.1547
2,Xác thực,700,20,0.0719,-0.0166,0.1387
3,Kiểm thử,123,5,0.0016,-0.1964,0.1880
4,Kiểm thử,123,10,0.0016,-0.1907,0.1397
5,Kiểm thử,123,20,0.0016,-0.1664,0.1245


- Khoảng bootstrap 95% của chênh lệch Sharpe đều chứa 0 với khối 5, 10 và 20 phiên.
- Dữ liệu hiện chưa cho thấy LightGBM có Sharpe tốt hơn quy tắc đảo chiều trong phiên.

## Mô phỏng vào và thoát vị thế trên tập kiểm thử

Mô phỏng theo từng mã trên 130 phiên tín hiệu, với tối đa 200 vị thế mua và 200 vị thế bán mỗi phiên.

In [5]:
coverage = pd.read_csv(results / "execution_coverage.csv").set_index("Model")
orders = pd.read_csv(results / "execution_orders.csv", parse_dates=["Planned exit", "Exit"])
coverage["Closed late"] = orders["Exit"].gt(orders["Planned exit"]).groupby(orders["Model"]).sum()
assert coverage["Orders"].eq(coverage[["Filled", "Unfilled", "Blocked"]].sum(axis=1)).all()
assert coverage["Deferred"].eq(coverage["Closed late"] + coverage["Unresolved"]).all()
display(coverage[["Orders", "Filled", "Unfilled", "Blocked", "Deferred", "Closed late", "Unresolved"]]
        .rename(index=model_labels, columns={"Orders": "Dự kiến", "Filled": "Đã vào",
            "Unfilled": "Thiếu giá vào", "Blocked": "Bị chặn", "Deferred": "Quá hạn thoát",
            "Closed late": "Đã thoát trễ", "Unresolved": "Còn mở cuối kỳ",
        }))

,Dự kiến,Đã vào,Thiếu giá vào,Bị chặn,Quá hạn thoát,Đã thoát trễ,Còn mở cuối kỳ
Model,,,,,,,
Ridge với 6 đặc trưng kỹ thuật,52000,51671,114,215,264,260,4
LightGBM giá và công bố tài chính,52000,51935,41,24,49,48,1


- LightGBM mở được 51.935 trong số 52.000 vị thế dự kiến. Các lệnh còn lại thiếu giá vào hoặc bị chặn.
- Trong 49 vị thế quá hạn thoát, 48 đã đóng sau `t+2` và 1 còn mở cuối kỳ.

## Kiểm tra đầu ra

In [6]:
ranks = pd.read_csv(results / "predictions.csv")
verification = json.loads((results / "reproduction.json").read_text(encoding="utf-8"))
assert verification["exact_ranks"]
assert ranks["Rank"].eq(ranks.groupby("Date").cumcount()).all()
display(ranks.head().round({"Score": 6}))
print(f"Sinh {len(ranks):,} hạng trên {ranks['Date'].nunique()} phiên. Thứ hạng khớp khi dựng lại từ dữ liệu gốc.")


,Date,SecuritiesCode,Score,Rank
0,2021-12-09,3193,0.002806,0
1,2021-12-09,2198,0.002639,1
2,2021-12-09,1873,0.002538,2
3,2021-12-09,9603,0.002482,3
4,2021-12-09,3491,0.002145,4


Sinh 263,081 hạng trên 132 phiên. Thứ hạng khớp khi dựng lại từ dữ liệu gốc.


- Hạng 0 là mã có điểm cao nhất ở mỗi phiên. Bảng đầu ra khớp khi tái lập từ dữ liệu gốc.
- Hai phiên cuối vẫn sinh hạng. Phép chấm chỉ dùng những phiên có mốc `t+2` trong dữ liệu.